In [ ]:
# Test GPU

In [23]:
!nvidia-smi


Fri Jul 17 01:05:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.167.08             Driver Version: 580.167.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
|  0%   28C    P8              3W /  450W |   16268MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Pobieranie qwen

In [10]:
pip install transformers accelerate qwen-vl-utils pillow

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install torch --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [19]:
import json
import time
from pathlib import Path
 
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Konfiguracja pliku

In [13]:
FOLDER_DOKUMENTOW = "./dokumenty_testowe"
PLIK_WYNIKOW = "./wyniki_qwen25vl.json"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
 
SCHEMA_POL = {
    "nazwa_wierzyciela": "",
    "krs": "",
    "nip": "",
    "regon": "",
    "adres_wierzyciela": "",
    "waluta": "",
    "wysokosc_wierzytelnosci": "",
    "termin_zaplaty": "",
    "sporna": "true/false",
    "zakres_podstawa_sporu": "",
    "uwagi": ""
}
 
INSTRUKCJA_EKSTRAKCJI = """Przeanalizuj ten dokument sądowy dotyczący wierzytelności i wyciągnij poniższe dane.
Zwróć WYŁĄCZNIE poprawny JSON, bez żadnego dodatkowego tekstu, komentarzy ani markdown code-fence.
 
Pola do wyciągnięcia:
- nazwa_wierzyciela: pełna nazwa/firma wierzyciela
- krs: numer KRS jeśli występuje
- nip: numer NIP jeśli występuje
- regon: numer REGON jeśli występuje
- adres_wierzyciela: pełny adres wierzyciela
- waluta: waluta wierzytelności (np. PLN, EUR)
- wysokosc_wierzytelnosci: kwota wierzytelności (sama liczba + waluta)
- termin_zaplaty: data lub termin zapłaty jeśli podany
- sporna: true jeśli z dokumentu wynika że wierzytelność jest sporna/kwestionowana, false jeśli nie
- zakres_podstawa_sporu: jeśli sporna=true, opisz na czym polega spór i w jakim zakresie; jeśli nie, wstaw null
- uwagi: wszelkie dodatkowe istotne informacje (typ dokumentu, prawomocność, braki formalne itp.)
 
Jeśli pole nie występuje w dokumencie, wstaw null (nie zgaduj)."""
 
 
def zbuduj_prompt_json():
    return INSTRUKCJA_EKSTRAKCJI + "\n\nFormat wyjściowy (przykładowa struktura):\n" + json.dumps(
        SCHEMA_POL, ensure_ascii=False, indent=2
    )
 

# Model

In [ ]:
def zaladuj_model():
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    return model, processor
 
 
def uruchom_qwen(model, processor, sciezka_obrazu: str) -> dict:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sciezka_obrazu},
                {"type": "text", "text": zbuduj_prompt_json()}
            ]
        }
    ]
 
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to("cuda")
 
    czas_start = time.time()
    output_ids = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    czas_trwania = time.time() - czas_start
 
    output_text = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
 
    return {"surowy_output": output_text, "czas_s": round(czas_trwania, 2)}

# Parsowanie tekstu

In [14]:
def sprobuj_sparsowac_json(tekst: str):
    tekst = tekst.strip()
    if tekst.startswith("```"):
        tekst = tekst.strip("`")
        if tekst.startswith("json"):
            tekst = tekst[4:]
    start = tekst.find("{")
    end = tekst.rfind("}")
    if start == -1 or end == -1:
        return None, "brak_json_w_odpowiedzi"
    try:
        return json.loads(tekst[start:end + 1]), None
    except json.JSONDecodeError as e:
        return None, f"blad_parsowania: {e}"
 

# Ladowanie Modelu

In [20]:
def zaladuj_model():
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    return model, processor
 
def uruchom_qwen(model, processor, sciezka_obrazu: str) -> dict:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sciezka_obrazu},
                {"type": "text", "text": zbuduj_prompt_json()}
            ]
        }
    ]
 
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to("cuda")
 
    czas_start = time.time()
    output_ids = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    czas_trwania = time.time() - czas_start
 
    output_text = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
 
    return {"surowy_output": output_text, "czas_s": round(czas_trwania, 2)}

# MAIN
## Czyszczenie pamieci

In [27]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def main():
    folder = Path(FOLDER_DOKUMENTOW)
    pliki = sorted([p for p in folder.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
 
    if not pliki:
        print(f"Brak plików .jpg/.png w {FOLDER_DOKUMENTOW} — wrzuć tam dokumenty testowe.")
        return
 
    print("Ładowanie modelu Qwen2.5-VL-7B...")
    model, processor = zaladuj_model()
    print("Model załadowany.\n")
 
    wyniki = []
 
    for plik in pliki:
        print(f"Przetwarzanie: {plik.name}")
        raw = uruchom_qwen(model, processor, str(plik))
        parsed, blad = sprobuj_sparsowac_json(raw["surowy_output"])
 
        wyniki.append({
            "plik": plik.name,
            "json": parsed,
            "blad_parsowania": blad,
            "czas_s": raw["czas_s"],
            "surowy_output": raw["surowy_output"],
        })
 
        with open(PLIK_WYNIKOW, "w", encoding="utf-8") as f:
            json.dump(wyniki, f, ensure_ascii=False, indent=2)
 
    print(f"\nGotowe. Wyniki zapisane w {PLIK_WYNIKOW}")
 
 
if __name__ == "__main__":
    main()

Ładowanie modelu Qwen2.5-VL-7B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model załadowany.

Przetwarzanie: 1646-2298-max.jpg
